# 20. Concurrency: Threading, Multiprocessing & AsyncIO: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **20. Concurrency: Threading, Multiprocessing & AsyncIO**. Python offers three distinct concurrency paradigms: multithreading (`concurrent.futures.ThreadPoolExecutor`) for I/O-bound tasks, multiprocessing (`ProcessPoolExecutor`) for CPU-bound tasks bypassing the GIL, and cooperative asynchronous programming (`asyncio`) for high-concurrency event-driven network I/O.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Multi-Threading: `threading.Thread`
- [x] 🔹 Thread Synchronization: `threading.Lock`
- [x] 🔹 Thread Pool Executor: `ThreadPoolExecutor`
- [x] 🔹 Process Pool Executor: `ProcessPoolExecutor`
- [x] 🔹 Asynchronous Coroutines: `async def` & `await`
- [x] 🔹 Event Loop Execution: `asyncio.run()`
- [x] 🔹 Concurrent Task Gathering: `asyncio.gather()`


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol, Literal, Final, TypedDict, Callable, TypeVar

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from ../data/raw_transactions.csv


### 🔹 Multi-Threading: `threading.Thread`
- **What it does:** Spawns OS threads sharing Python heap memory space (suitable for I/O-bound tasks).
- **Syntax:** `threading.Thread`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Applies Multi-Threading on fintech records using columns `transaction_id` to demonstrate real-world execution.


In [2]:
def worker_audit(tx_id):
    print(f'Thread auditing {tx_id}')

t1 = threading.Thread(target=worker_audit, args=(transactions[0]['transaction_id'],))
t1.start()
t1.join()
print('Thread worker finished.')

Thread auditing TX109326
Thread worker finished.


### 🔹 Thread Synchronization: `threading.Lock`
- **What it does:** Mutual exclusion lock preventing race conditions on shared memory across threads.
- **Syntax:** `threading.Lock`
  - **Parameters:**
    - `row_indexer` (*scalar, slice, list, or boolean mask*): Row identifier(s).
  - **Optional Parameters:**
    - `col_indexer` (*scalar, slice, list, or boolean mask*): Column identifier(s).
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Applies Thread Synchronization on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [3]:
total_balance = 0.0
lock = threading.Lock()

def thread_safe_deposit(amt):
    global total_balance
    with lock:
        total_balance += amt

threads = [threading.Thread(target=thread_safe_deposit, args=(float(transactions[i]['transaction_amount']),)) for i in range(3)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Thread-Safe Consolidated Balance: ${total_balance:,.2f}')

Thread-Safe Consolidated Balance: $2,490.97


### 🔹 Thread Pool Executor: `ThreadPoolExecutor`
- **What it does:** High-level thread pool manager mapping worker functions across threads.
- **Syntax:** `ThreadPoolExecutor`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Applies Thread Pool Executor on fintech records using columns `card_type`, `transaction_id` to demonstrate real-world execution.


In [4]:
from concurrent.futures import ThreadPoolExecutor
def process_item(t):
    return f"{t['transaction_id']}: Verified {t['card_type']}"

with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(process_item, transactions[:3]))
print('ThreadPoolExecutor results:', results)

ThreadPoolExecutor results: ['TX109326: Verified Visa', 'TX106376: Verified Visa', 'TX103301: Verified Visa']


### 🔹 Process Pool Executor: `ProcessPoolExecutor`
- **What it does:** Spawns independent OS Python processes bypassing the GIL for CPU-bound computation.
- **Syntax:** `ProcessPoolExecutor`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Demonstrates Process Pool Executor with practical fintech data structures and variables in the following code block.


In [5]:
from concurrent.futures import ProcessPoolExecutor
print('ProcessPoolExecutor: Bypasses CPython GIL by spawning separate OS processes.')

ProcessPoolExecutor: Bypasses CPython GIL by spawning separate OS processes.


### 🔹 Asynchronous Coroutines: `async def` & `await`
- **What it does:** Defines non-blocking coroutines yielding control back to event loop on async I/O.
- **Syntax:** `async def`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Demonstrates Asynchronous Coroutines with practical fintech data structures and variables in the following code block.


In [6]:
async def simulate_async_api(tx_id):
    await asyncio.sleep(0.01) # Non-blocking async sleep
    return f'{tx_id}: APPROVED'

print('Coroutine function defined.')

Coroutine function defined.


### 🔹 Event Loop Execution: `asyncio.run()`
- **What it does:** Creates a new event loop, executes the main coroutine, and closes the loop.
- **Syntax:** `asyncio.run()`
- **Key Note:** Always remember to include `self` as the first parameter in instance methods so Python knows which object instance is executing.
- **Dataset Application & Code Demonstration:** Applies Event Loop Execution on fintech records using columns `transaction_id` to demonstrate real-world execution.


In [7]:
import nest_asyncio
nest_asyncio.apply()
async def main():
    return await simulate_async_api(transactions[0]['transaction_id'])
print('asyncio.run result:', asyncio.run(main()))


asyncio.run result: TX109326: APPROVED


### 🔹 Concurrent Task Gathering: `asyncio.gather()`
- **What it does:** Runs multiple async coroutines concurrently on single-threaded event loop.
- **Syntax:** `asyncio.gather()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Applies Concurrent Task Gathering on fintech records using columns `transaction_id` to demonstrate real-world execution.


In [8]:
import nest_asyncio
nest_asyncio.apply()
async def run_batch():
    tasks = [simulate_async_api(t['transaction_id']) for t in transactions[:3]]
    return await asyncio.gather(*tasks)
print('asyncio.gather results:', asyncio.run(run_batch()))


asyncio.gather results: ['TX109326: APPROVED', 'TX106376: APPROVED', 'TX103301: APPROVED']


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: When to choose Threading vs Multiprocessing vs Asyncio
- **Objective:** Q1: When to choose Threading vs Multiprocessing vs Asyncio
- **Approach:** Explain trade-offs: (1) Threading for I/O blocking calls; (2) Multiprocessing for CPU bound GIL bypass; (3) Asyncio for high concurrency non-blocking network sockets.
- **Syntax:** `ProcessPoolExecutor` vs `ThreadPoolExecutor` vs `asyncio`

In [9]:
print('CPU-Bound: Multiprocessing (separate OS processes).')
print('I/O-Bound High Concurrency: Asyncio (event loop coroutines).')
print('I/O-Bound Blocking APIs: Threading.')

CPU-Bound: Multiprocessing (separate OS processes).
I/O-Bound High Concurrency: Asyncio (event loop coroutines).
I/O-Bound Blocking APIs: Threading.
